# TEK Runtime — Kaggle Notebook
**Temporary Environment on Kaggle** — QEMU virtual machine with secure remote access.

## Setup checklist
1. In **TEK Controller** (Windows): `Connection → Generate Secret → Copy`.
2. Add a Kaggle Secret named `TEK_SESSION_SECRET` with the copied value.
   *(Add-ons → Secrets → New Secret)*.
3. Run all cells in order.
4. The notebook will print:
   ```
   host = abc-xyz.trycloudflare.com
   pair = ABCDEF
   ```
5. Verify `pair` matches the controller, enter `host`, click **Fetch & Connect**.

> ⚠️  Never print `TEK_SESSION_SECRET` or any derived token.
> ⚠️  Re-running individual cells is safe — each cell is idempotent where possible.


In [ ]:
# ── M0: Kaggle host diagnostics ──────────────────────────────────────
# Record CPU / RAM / Disk / KVM availability.
# Results here are EVIDENCE. Do not assume any value.
import os, shutil, math

def _read(path, default='unavailable'):
    try: return open(path).read().strip()
    except OSError: return default

def _bytes(n): return f'{n / (1024**3):.2f} GB'

# CPU
try:
    cpu_affinity = len(os.sched_getaffinity(0))
except AttributeError:
    cpu_affinity = os.cpu_count() or 1

cpu_cgroup_v2 = _read('/sys/fs/cgroup/cpu.max', 'max 100000')
cpu_model     = _read('/proc/cpuinfo').split('model name')[-1].split('\n')[0].lstrip(': ') or 'unknown'

# RAM
ram_avail_kb  = 0
for line in open('/proc/meminfo'):
    if line.startswith('MemAvailable:'):
        ram_avail_kb = int(line.split()[1])
ram_total_kb  = 0
for line in open('/proc/meminfo'):
    if line.startswith('MemTotal:'):
        ram_total_kb = int(line.split()[1])

cgroup_ram = _read('/sys/fs/cgroup/memory.max', 'max')

# Disk
scratch = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
du = shutil.disk_usage(scratch)

# KVM
kvm_available = os.path.exists('/dev/kvm')

# QEMU version
import subprocess
qemu_ver = subprocess.run(['qemu-system-x86_64', '--version'],
    capture_output=True, text=True).stdout.splitlines()[0] \
    if shutil.which('qemu-system-x86_64') else 'NOT INSTALLED'

print('═' * 60)
print('M0: KAGGLE HOST DIAGNOSTICS')
print('═' * 60)
print(f'CPU affinity     : {cpu_affinity} vCPU')
print(f'CPU cgroup.max   : {cpu_cgroup_v2}')
print(f'CPU model        : {cpu_model[:60]}')
print(f'RAM total        : {_bytes(ram_total_kb * 1024)}')
print(f'RAM available    : {_bytes(ram_avail_kb * 1024)}')
print(f'RAM cgroup limit : {cgroup_ram}')
print(f'Disk free ({scratch}) : {_bytes(du.free)}')
print(f'KVM available    : {kvm_available}')
print(f'QEMU             : {qemu_ver}')
print('═' * 60)


In [ ]:
# ── PRIVATE: load TEK_SESSION_SECRET ─────────────────────────────────
# This cell reads the secret from Kaggle Secrets.
# NEVER print or log TEK_SESSION_SECRET.
import os
try:
    from kaggle_secrets import UserSecretsClient
    _secret = UserSecretsClient().get_secret('TEK_SESSION_SECRET')
    if not _secret:
        raise ValueError('Secret is empty')
    os.environ['TEK_SESSION_SECRET'] = _secret
    del _secret   # don't leave it in a named variable
    print('[SECRET] TEK_SESSION_SECRET loaded from Kaggle Secrets.')
except Exception as exc:
    # Fallback: check if it was set in a previous cell
    if os.environ.get('TEK_SESSION_SECRET'):
        print('[SECRET] TEK_SESSION_SECRET already set in environment.')
    else:
        raise RuntimeError(
            'TEK_SESSION_SECRET is not set.\n'
            'Add it via Add-ons → Secrets → New Secret (name: TEK_SESSION_SECRET)'
        ) from exc


In [ ]:
# ── Install TEK Runtime and dependencies ─────────────────────────────
# Idempotent: re-running this cell is safe.
import subprocess, sys

def _run(cmd, **kw):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}\n{r.stderr[-1000:]}')
    return r.stdout

print('Installing system packages...')
_run('apt-get -qq update')
_run('DEBIAN_FRONTEND=noninteractive apt-get -qq install -y '
     'qemu-system-x86 qemu-utils novnc websockify')

print('Installing cloudflared...')
import os, stat
if not os.path.exists('/usr/local/bin/cloudflared'):
    _run('curl -fsSL -o /usr/local/bin/cloudflared '
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.chmod('/usr/local/bin/cloudflared',
             stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)

print('Installing tek-runtime...')
_run(f'{sys.executable} -m pip install -q '
     '"git+https://github.com/ALI88708/TEK.git#subdirectory=packages/tek-runtime"')

# Verify
import tek_runtime
print(f'tek-runtime {tek_runtime.__version__} installed OK')


In [ ]:
# ── M0b: Storage validation ──────────────────────────────────────────
# Write a known pattern, read back, verify. Proves storage is writable
# and not silently dropping data.
import os, hashlib

scratch = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
test_file = os.path.join(scratch, 'tek_storage_test.bin')
PATTERN = b'\x5a' * (4 * 1024 * 1024)  # 4 MB of 0x5A
expected = hashlib.sha256(PATTERN).hexdigest()

with open(test_file, 'wb') as f:
    f.write(PATTERN)

with open(test_file, 'rb') as f:
    actual_sha = hashlib.sha256(f.read()).hexdigest()

os.remove(test_file)

assert actual_sha == expected, f'Storage readback mismatch! {actual_sha} != {expected}'
print(f'M0b PASS: 4 MB write/readback verified (sha256={expected[:16]}...)')


In [ ]:
# ── Fetch Alpine Linux ISO ────────────────────────────────────────────
# Uses a cached copy if already downloaded in this session.
from tek_runtime.fetch_iso import get_alpine
iso_path = get_alpine()
print(f'ISO: {iso_path}')

# SHA-256 verification (Alpine publishes sha256 files)
import hashlib, urllib.request, os
import re as _re

_BASE = 'https://dl-cdn.alpinelinux.org/alpine/latest-stable/releases/x86_64/'
_iso_name = os.path.basename(iso_path)
try:
    sha_url  = _BASE + _iso_name + '.sha256'
    expected = urllib.request.urlopen(sha_url, timeout=15).read().decode().split()[0]
    print(f'Expected SHA-256: {expected}')
    actual = hashlib.sha256(open(iso_path, 'rb').read()).hexdigest()
    print(f'Actual   SHA-256: {actual}')
    if actual == expected:
        print('ISO integrity: PASS')
    else:
        raise RuntimeError(f'ISO SHA-256 mismatch! Expected {expected}, got {actual}')
except Exception as exc:
    print(f'ISO integrity check skipped or failed: {exc}')


In [ ]:
# ── Start TEK session ────────────────────────────────────────────────
# This cell starts QEMU, the auth proxy, and the tunnel.
# It blocks until the session ends.
#
# Option 1 (default): Alpine Linux auto-downloaded
from tek_runtime import main
main(iso_path=iso_path)

# Option 2: Custom Linux ISO from a Kaggle dataset
# from tek_runtime import main
# main(iso_path='/kaggle/input/my-iso-dataset/ubuntu-24.04.iso')

# Option 3: Full control via GuestConfig
# from tek_runtime import main, GuestConfig
# main(guest=GuestConfig.for_custom_linux(
#     '/kaggle/input/my-iso-dataset/debian.iso',
#     display_name='Debian 12',
# ))

# To stop from another cell while this is running:
# from tek_runtime import stop_active_session
# stop_active_session()
